In [2]:
from typing import List, TypedDict, Tuple, Dict
from dataclasses import dataclass

from typing import List, TypedDict, Tuple, Dict
from dataclasses import dataclass
from services.bedrock import BedrockClaude3Model
import structlog
import boto3

In [3]:
@dataclass
class Document():
    name: str = None
    num_pages: int = 0
    total_num_words: int = 0
    text: List[Dict[str, str]] = None
    summary: str = None

In [4]:
doc_1 = Document(name="name")

In [5]:
@dataclass
class ModelMetadata:
    input_tokens: int
    output_tokens: int


@dataclass
class DocumentPage():
    page_number: int = 0
    num_words: int = 0
    text: str = None


@dataclass
class Document():
    name: str = None
    num_pages: int = 0
    total_num_words: int = 0
    text: List[DocumentPage] = None
    summary: str = None

In [6]:
doc_1.num_pages = 10

In [12]:
def run_model(
    model: BedrockClaude3Model, prompt: str
) -> Tuple[Dict | None, ModelMetadata]:
    """Run the LLM model with the given document and prompt. Return values even if model fails."""
    try:
        response = model.run(query=prompt)
    except Exception as e:
        logger.exception(f"Error running llm extraction for prompt: {prompt}")
        result = None
        metadata = ModelMetadata(input_tokens=0, output_tokens=0)
        return result, metadata
    
    result = response.response[0]
    
    if "input" in result:
        final_result = result["input"]
    elif "text" in result:
        final_result = result["text"]
    else:
        raise ValueError("Unexpected response format from LLM model.")
    requirements_metadata = ModelMetadata(
        input_tokens=response.metadata.get("input_tokens", 0),
        output_tokens=response.metadata.get("output_tokens", 0),
    )
    return final_result, requirements_metadata

In [13]:
import boto3
boto3.setup_default_session(profile_name='arcanum-dev')

In [14]:
output, tokens = run_model(model=BedrockClaude3Model(), prompt="Hello")

In [15]:
output

'Hello! How can I assist you today? Feel free to ask me any questions or let me know if you need help with anything.'

In [ ]:
output

In [11]:
BedrockClaude3Model().run(query="How are you?")

GPTResponse(response=[{'type': 'text', 'text': "As an AI language model, I don't have feelings or personal experiences, but I'm functioning well and ready to assist you with any questions or tasks you may have. How can I help you today?"}], metadata={'input_tokens': 11, 'output_tokens': 44})